In [ ]:
import pandas as pd
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (classification_report, roc_curve, auc,
                                     precision_score, recall_score, f1_score)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

d:\ML Projects\El_Farghaly_Bros\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
path = kagglehub.dataset_download("vivekvivek13/bank-customers-prediction")
df = pd.read_csv(f"{path}/bank.csv", delimiter=';')
df

100%|██████████| 391k/391k [00:00<00:00, 785kB/s]

Extracting files...


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


In [ ]:
X = df.drop(columns = 'y')
y = df.y

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = .2,
    stratify = y,
    random_state = 30
)

print(f"Train size : {X_train.shape}")
print(f"Test size  : {X_test.shape}")
print(f"Class dist : {np.bincount(y_test)}")

ohe_cols = X.select_dtypes('object').columns
ordinal_cols = []
num_cols = X.select_dtypes('number').columns

ohe = OneHotEncoder(drop = 'first', sparse_output = False)
scaler = StandardScaler()

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
X_train[ohe_cols] = ohe.fit_transform(X_train[ohe_cols])
X_test[ohe_cols] = ohe.transform(X_test[ohe_cols])

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


In [ ]:
vif_data = pd.DataFrame()
vif_data.Feature = X.columns
vif_data.VIF = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values(by = 'VIF', ascending = False)
print(vif_data)

# Modeling

In [ ]:
# CONFIG
random_state = 30
n_splits     = 5
n_trials     = 50
scoring      = "recall"
thresholds = [.2, .25, .3, .35, .4, .45, .5]
results    = []

skf = StratifiedKFold(n_splits = n_splits, shuffle = True, random_state = random_state)

# Shared dict — each model block writes its proba here
test_proba = {}

## Logistic Regression

In [ ]:
# Optuna objective
def objective_lr(trial):
    params = {
        "C"           : trial.suggest_float("C", 0.001, 10.0, log = True),
        "max_iter"    : trial.suggest_int("max_iter", 200, 1000),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
    }
    model = LogisticRegression(**params, random_state = 30)
    # CV is run on the already-scaled train set
    return cross_val_score(model, X_train, y_train,
                           cv= skf, scoring= scoring, n_jobs=-1).mean()

study_lr = optuna.create_study(
    direction="maximize",
    sampler = optuna.samplers.TPESampler(seed = random_state)
)
study_lr.optimize(objective_lr, n_trials= n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_lr.best_value:.3f}")
print(f"  Best params      : {study_lr.best_params}")

# Refit on full train set with best params
lr_best = LogisticRegression(**study_lr.best_params, random_state = random_state)
lr_best.fit(X_train, y_train)

lr_proba = lr_best.predict_proba(X_test)[:, 1]
test_proba["Logistic Regression"] = lr_proba   # save for ROC-AUC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, lr_proba, digits = 2))
print("-"*55)

In [ ]:
# Threshold optimisation

for t in thresholds:
    preds = (lr_proba >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division = 0),
        "recall"   : recall_score(y_test,    preds, zero_division = 0),
        "f1"       : f1_score(y_test,        preds, zero_division = 0),
    })

df_lr_thresh = pd.DataFrame(results)
best_lr = df_lr_thresh.loc[df_lr_thresh["recall"].idxmax()]

print("-"*55)
print("  Threshold optimization  (maximise Recall)")
print("-"*55)
print(f"  Optimal threshold : {best_lr['threshold']:.2f}")
print(f"  Precision         : {best_lr['precision']:.2f}")
print(f"  Recall            : {best_lr['recall']:.2f}")
print(f"  F1                : {best_lr['f1']:.2f}")

print("\n  Classification report at optimal threshold:")
lr_preds_opt = (lr_proba >= best_lr["threshold"]).astype(int)
print(classification_report(y_test, lr_preds_opt, digits=4))


df_plot = df_lr_thresh.melt(id_vars = 'threshold', 
                            value_vars = ['precision', 'recall', 'f1'],
                            var_name = 'Metric', 
                            value_name = 'Score')

# Precision / Recall curve for this model
plt.figure(figsize=(10, 5))
ax = sns.lineplot(data = df_plot,
                  x = 'threshold',
                  y = 'Score',
                  hue = 'Metric', 
                  palette = {'precision': '#4A90D9', 'recall': '#E74C3C', 'f1': '#27AE60'})
plt.axvline(best_lr["threshold"], color="gray", linestyle="--", 
            label=f"Optimal t={best_lr['threshold']:.3f}")

plt.title("Logistic Regression — Threshold vs Metrics", fontsize=14, pad=15)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend(title="Metrics", frameon=True)
plt.tight_layout()
plt.show()

## Random Forest Classifier

In [ ]:
# Optuna objective
def objective_rf(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"        : trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features"     : trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight"     : trial.suggest_categorical("class_weight", [None, "balanced"]),
    }
    model = RandomForestClassifier(**params, random_state = random_state, n_jobs=-1)
    return cross_val_score(model, X_train, y_train,
                           cv = skf, scoring = scoring, n_jobs=-1).mean()

study_rf = optuna.create_study(
    direction = "maximize",
    sampler=optuna.samplers.TPESampler(seed = random_state)
)
study_rf.optimize(objective_rf, n_trials = n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_rf.best_value:.3f}")
print(f"  Best params      : {study_rf.best_params}")

# Refit on full train set with best params
rf_best = RandomForestClassifier(
    **study_rf.best_params, random_state = random_state, n_jobs=-1
)
rf_best.fit(X_train, y_train)

rf_proba = rf_best.predict_proba(X_test)[:, 1]
test_proba["Random Forest"] = rf_proba   # save for ROC-AUC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report")
print(classification_report(y_test, rf_proba, digits = 2))
print("-"*55)

In [ ]:
# Threshold optimisation
for t in thresholds:
    preds = (rf_proba >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall"   : recall_score(y_test,    preds, zero_division=0),
        "f1"       : f1_score(y_test,        preds, zero_division=0),
    })

df_rf_thresh = pd.DataFrame(results)
best_rf      = df_rf_thresh.loc[df_rf_thresh["recall"].idxmax()]

print("═"*55)
print("  THRESHOLD OPTIMISATION  (maximise Recall)")
print("═"*55)
print(f"  Optimal threshold : {best_rf['threshold']:.3f}")
print(f"  Precision         : {best_rf['precision']:.4f}")
print(f"  Recall            : {best_rf['recall']:.4f}")
print(f"  F1                : {best_rf['f1']:.4f}")

print("\n  Classification report at optimal threshold:")
rf_preds_opt = (rf_proba >= best_rf["threshold"]).astype(int)
print(classification_report(y_test, rf_preds_opt, digits = 3))

df_plot = df_rf_thresh.melt(id_vars = 'threshold', 
                            value_vars = ['precision', 'recall', 'f1'],
                            var_name = 'Metric', 
                            value_name = 'Score')

# Precision / Recall curve for this model
plt.figure(figsize=(10, 5))
ax = sns.lineplot(data = df_plot,
                  x = 'threshold',
                  y = 'Score',
                  hue = 'Metric', 
                  palette = {'precision': '#4A90D9', 'recall': '#E74C3C', 'f1': '#27AE60'})
plt.axvline(best_lr["threshold"], color="gray", linestyle="--", 
            label=f"Optimal t={best_lr['threshold']:.3f}")

plt.title("Random Forest — Threshold vs Metrics", fontsize=14, pad=15)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend(title="Metrics", frameon=True)
plt.tight_layout()
plt.show()

## XGBoost

In [ ]:
# Optuna objective
def objective_xgb(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"       : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.001, 0.3, log = True),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 0.0001, 10.0, log = True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 0.0001, 10.0, log = True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
    }
    model = XGBClassifier(
        **params, random_state = random_state,
        eval_metric="logloss", verbosity=0, n_jobs=-1
    )
    return cross_val_score(model, X_train, y_train,
                           cv = skf, scoring = scoring, n_jobs=-1).mean()

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed = random_state)
)
study_xgb.optimize(objective_xgb, n_trials = n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_xgb.best_value:.2f}")
print(f"  Best params      : {study_xgb.best_params}")

# Refit on full train set with best params
xgb_best = XGBClassifier(
    **study_xgb.best_params, random_state = random_state,
    eval_metric="logloss", verbosity=0, n_jobs=-1
)
xgb_best.fit(X_train, y_train)

xgb_proba = xgb_best.predict_proba(X_test)[:, 1]
test_proba["XGBoost"] = xgb_proba   # save for ROC-AUC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, xgb_proba, digits = 2))
print("-"*55)

In [ ]:
# Threshold optimisation
for t in thresholds:
    preds = (xgb_proba >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division = 0),
        "recall"   : recall_score(y_test,    preds, zero_division = 0),
        "f1"       : f1_score(y_test,        preds, zero_division = 0),
    })

df_xgb_thresh = pd.DataFrame(results)
best_xgb      = df_xgb_thresh.loc[df_xgb_thresh["recall"].idxmax()]

print("═"*55)
print("  Threshold Optimization  (maximise Recall)")
print("═"*55)
print(f"  Optimal threshold : {best_xgb['threshold']:.2f}")
print(f"  Precision         : {best_xgb['precision']:.2f}")
print(f"  Recall            : {best_xgb['recall']:.2f}")
print(f"  F1                : {best_xgb['f1']:.2f}")

print("\n  Classification report at optimal threshold:")
xgb_preds_opt = (xgb_proba >= best_xgb["threshold"]).astype(int)
print(classification_report(y_test, xgb_preds_opt, digits = 3))

df_plot = df_xgb_thresh.melt(id_vars = 'threshold', 
                            value_vars = ['precision', 'recall', 'f1'],
                            var_name = 'Metric', 
                            value_name = 'Score')

# Precision / Recall curve for this model
plt.figure(figsize=(10, 5))
ax = sns.lineplot(data = df_plot,
                  x = 'threshold',
                  y = 'Score',
                  hue = 'Metric', 
                  palette = {'precision': '#4A90D9', 'recall': '#E74C3C', 'f1': '#27AE60'})
plt.axvline(best_lr["threshold"], color="gray", linestyle="--", 
            label=f"Optimal t={best_lr['threshold']:.3f}")

plt.title("Random Forest — Threshold vs Metrics", fontsize=14, pad=15)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend(title="Metrics", frameon=True)
plt.tight_layout()
plt.show()

## LightGBM

In [ ]:
# Optuna objective
def objective_lgbm(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"       : trial.suggest_int("max_depth", 3, 12),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.001, 0.3, log = True),
        "num_leaves"      : trial.suggest_int("num_leaves", 20, 150),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 0.0001, 10.0, log = True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 0.0001, 10.0, log = True),
        "class_weight"    : trial.suggest_categorical("class_weight", [None, "balanced"]),
    }
    model = LGBMClassifier(**params, random_state = random_state, n_jobs=-1, verbose=-1)
    return cross_val_score(model, X_train, y_train,
                           cv = skf, scoring = scoring, n_jobs=-1).mean()

study_lgbm = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed = random_state)
)
study_lgbm.optimize(objective_lgbm, n_trials = n_trials, show_progress_bar = True)

print(f"\n✔ Best CV {scoring} : {study_lgbm.best_value:.3f}")
print(f"  Best params      : {study_lgbm.best_params}")

# Refit on full train set with best params
lgbm_best = LGBMClassifier(
    **study_lgbm.best_params, random_state = random_state, n_jobs=-1, verbose=-1
)
lgbm_best.fit(X_train, y_train)

lgbm_proba = lgbm_best.predict_proba(X_test)[:, 1]
test_proba["LightGBM"] = lgbm_proba   # save for ROC plot

# Classification report
print("\n" + "-"*55)
print("  Classification Report:")
print(classification_report(y_test, lgbm_proba, digits = 2))
print("-"*55)

In [ ]:
# Threshold optimisation
for t in thresholds:
    preds = (lgbm_proba >= t).astype(int)
    results.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division = 0),
        "recall"   : recall_score(y_test,    preds, zero_division = 0),
        "f1"       : f1_score(y_test,        preds, zero_division = 0),
    })

df_lgbm_thresh = pd.DataFrame(results)
best_lgbm      = df_lgbm_thresh.loc[df_lgbm_thresh["recall"].idxmax()]

print("-"*55)
print("  Threshold Optimization  (maximise Recall)")
print("-"*55)
print(f"  Optimal threshold : {best_lgbm['threshold']:.3f}")
print(f"  Precision         : {best_lgbm['precision']:.3f}")
print(f"  Recall            : {best_lgbm['recall']:.3f}")
print(f"  F1                : {best_lgbm['f1']:.3f}")

print("\n  Classification report at optimal threshold:")
lgbm_preds_opt = (lgbm_proba >= best_lgbm["threshold"]).astype(int)
print(classification_report(y_test, lgbm_preds_opt, digits=4))

df_plot = df_lgbm_thresh.melt(id_vars = 'threshold', 
                            value_vars = ['precision', 'recall', 'f1'],
                            var_name = 'Metric', 
                            value_name = 'Score')

# Precision / Recall curve for this model
plt.figure(figsize=(10, 5))
ax = sns.lineplot(data = df_plot,
                  x = 'threshold',
                  y = 'Score',
                  hue = 'Metric', 
                  palette = {'precision': '#4A90D9', 'recall': '#E74C3C', 'f1': '#27AE60'})
plt.axvline(best_lr["threshold"], color="gray", linestyle="--", 
            label=f"Optimal t={best_lr['threshold']:.3f}")

plt.title("Random Forest — Threshold vs Metrics", fontsize=14, pad=15)
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend(title="Metrics", frameon=True)
plt.tight_layout()
plt.show()

## ROC-AUC Chart

In [ ]:
COLORS = {
    "Logistic Regression": "#4A90D9",
    "Random Forest": "#27AE60",
    "XGBoost": "#E74C3C",
    "LightGBM": "#F39C12",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("ROC-AUC Comparison Using Optuna Tuned Models", 
             fontsize=16, fontweight="bold", y=1.02)

# 1. LEFT: ROC Curves (Matplotlib + Seaborn Styling)
ax = axes[0]
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random classifier", alpha=0.5)

# Calculate AUCs for the leaderboard and plot simultaneously
rows = []
for name, proba in test_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    score = auc(fpr, tpr)
    rows.append({"Model": name, "AUC": score})
    
    # Plotting line
    sns.lineplot(x=fpr, y=tpr, ax=ax, lw=2.5, color=COLORS[name],
                 label=f"{name} (AUC = {score:.4f})")

ax.set_title("ROC Curves", fontsize=14, pad=10)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right", frameon=True)

# 2. RIGHT: AUC Bar Chart (Seaborn)
ax2 = axes[1]
df_aucs = pd.DataFrame(rows).sort_values("AUC", ascending=False)

sns.barplot(data=df_aucs, x='AUC', y='Model', palette=COLORS, 
            ax=ax2, hue='Model', legend=False, alpha=0.9, edgecolor="white")

# Add text labels inside the bars
for i, val in enumerate(df_aucs['AUC']):
    ax2.text(val - 0.005, i, f"{val:.4f}", va='center', ha='right', 
             color='white', fontweight='bold', fontsize=11)

ax2.set_title("AUC Score Comparison", fontsize=14, pad=10)
ax2.set_xlim([df_aucs['AUC'].min() - 0.02, 1.0])
ax2.set_xlabel("ROC-AUC Score")
ax2.set_ylabel("") # Remove 'Model' label for cleaner look

plt.tight_layout()
plt.savefig("roc_auc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Final Leaderboard ---
print("\n" + "="*40)
print(f"{'RANK':<5} {'MODEL':<20} {'TEST AUC':<10}")
print("-" * 40)
for i, row in enumerate(df_aucs.itertuples(), 1):
    print(f"{i:<5} {row.Model:<20} {row.AUC:<10.4f}")
print("="*40)